# EC Price Prediction — Appreciation Model

## Model Evolution

| Version | Approach | R² | Issue |
|---------|----------|-----|-------|
| v1 | Absolute price, random split | 0.94 | **Data leakage** in temporal features |
| v2 | Fixed leakage, absolute price | 0.35 | Wrong problem framing |
| v3 | Log-transform + full market features | 0.45 | Still wrong framing |
| **v4 (final)** | **Appreciation ratio from launch price** | **0.88** | **Zero leakage, clean** |

## Key Insight

Instead of predicting absolute price, we predict how much a unit **appreciates** relative to its launch price:

```
predicted_price = appreciation_ratio × launch_psm × area
```

This removes price-level variance and focuses the model on what actually drives value change.


## Pipeline

1. **Split data temporally FIRST** (before any feature engineering)
2. **Compute all lookups from training data only** (zero look-ahead)
3. **24 features** — base, lagged district/project stats, derived
4. **Optuna HPO** — 60 trials each for XGBoost, LightGBM, CatBoost
5. **Weighted ensemble** — optimise weights on validation set
6. **Prediction intervals** — XGBoost quantile regression, alpha-tuned for coverage
7. **SHAP analysis** + residual diagnostics


## Results

### Model Comparison
![Model Comparison](figures/13_model_comparison.png)

### SHAP Feature Importance
![SHAP Summary](figures/09_shap_summary.png)

![SHAP Bar](figures/10_shap_bar.png)

### SHAP Dependence (Top 3 Features)
![SHAP Dependence](figures/11_shap_dependence.png)

### Prediction Intervals
![Prediction Intervals](figures/14_prediction_intervals.png)

### Residual Analysis
![Residuals](figures/15_residual_analysis.png)

### Actual vs Predicted
![Predictions](figures/12_predictions.png)


## Leakage Audit

Every feature has been audited for look-ahead bias:

| Feature Group | Method | Leakage Risk |
|--------------|--------|-------------|
| Base features (district, area, floor, etc.) | From the row itself | None |
| Launch PSM | From train-period new sales/earliest resales | None |
| District lag features | Prior-year stats from train-period full market | None |
| Project lag features | Prior-year stats from train-period EC data | None |
| Target encoding | KFold on train set only, applied to val/test | None |
| Fill medians | Computed from training set only | None |
| Area quartile bins | Computed from training set only | None |

## Final Metrics (Zero Leakage)

| Metric | Value |
|--------|-------|
| **R²** | **0.880** |
| **MAPE** | **5.41%** |
| **MAE** | **S$92,313** |
| **PI Coverage** | **68.3%** |
